In [7]:
import argparse
import os
import random
import shutil
import warnings
from pathlib import Path

import numpy as np
import torch
import yaml

import pandas as pd
from speakleash import Speakleash

from utils.download_huggingface_data import download_hf_data
from utils.download_speakleash_data import save_quality_docs
from utils.download_spokes_data import (
    change_xls_to_txt,
    download_conversations_transcripts,
)
from utils.preprocess_files import get_files_names, preprocess_files
from utils.train_test_split import create_train_test_split
from utils.hyperparameters_optimization import run_hyperparameters_optimization
from utils.ngram_lstm_compare import LSTMPredictor, ModelEvaluator

from src.model import train_model
from src.train_tokenizer import train_tokenizers
from src.fine_tuning import fine_tune_model


warnings.filterwarnings("ignore")

In [27]:
# paths in config.yml are relative to the project root
PROJECT_ROOT = Path(os.path.abspath("."))

PROJECT_CONFIG_PATH = PROJECT_ROOT / "config/config.yml"
MODEL_CONFIG_PATH = "config/model_config.yml"
STUDY_RESULTS_DIR = "study_results"

MODEL_CHECKPOINT_PATH = "model_checkpoint.pt"
# name of the best checkpoint is set by src/model.py and src/fine_tuning.py
BEST_MODEL_CHECKPOINT_PATH = "model_checkpoint_best.pt"
BEST_FINE_TUNED_CHECKPOINT_PATH = "fined_tuned_model_checkpoint_best.pt"

# file names expected by src/model_loader.py
MODEL_PATH = "model.pt"
FINE_TUNED_MODEL_PATH = "fine_tuned_model.pt"
TOKENIZER_PATH = "spm_pl.model"

# beam search parameters used for the evaluation
BEAM_WIDTH = 25
MAX_WORD_LENGTH = 10
ALPHA = 0.2

In [11]:
print(torch.cuda.is_available())
torch.cuda.get_device_name(0)

False
/home/julia/Desktop/Next_token_prediction_model/venv/bin/python


In [16]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def load_config(config_path: str) -> dict:
    with open(config_path) as f:
        return yaml.safe_load(f)


def extract_state_dict(checkpoint_path: str, model_path: str) -> None:
    """Save only the weights from a training checkpoint."""
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    torch.save(checkpoint["model_state_dict"], model_path)
    print(f"Saved model from {checkpoint_path} to {model_path}")


def download(config: dict) -> None:
    # downloading speakleash datasets
    Path(config["dir-path"]).mkdir(parents=True, exist_ok=True)
    sl = Speakleash(config["dir-path"])
    save_quality_docs(
        sl,
        dir_path=config["dir-path"],
        datasets_names=config["speakleash-datasets"],
        limit=config["data-limit"],
    )

    # downloading huggingface news dataset
    download_hf_data(
        url=config["huggingface-dataset-url"],
        dir_path=config["dir-path"],
        dataset_name=config["huggingface-dataset-name"],
    )

    # downloading spokes datasets
    conversations = pd.read_excel(config["conversations-id-file"])
    conversations_train = conversations[conversations["Id"].str.startswith("CBIZ")]
    conversations_test = conversations[~conversations["Id"].str.startswith("CBIZ")]

    print("Saving train conversations ... ")
    for file_path in download_conversations_transcripts(
        conversations_df=conversations_train,
        dir_path=config["dialog-dir-path"],
        dataset_name=config["conversations-train-dir-name"],
    ):
        change_xls_to_txt(file_path)

    print("Saving test conversations ... ")
    for file_path in download_conversations_transcripts(
        conversations_df=conversations_test,
        dir_path=config["dialog-dir-path"],
        dataset_name=config["conversations-test-dir-name"],
    ):
        change_xls_to_txt(file_path)


def preprocess(config: dict) -> None:
    dir_path = config["dir-path"]
    preprocessed_dir_path = config["preprocessed-dir-path"]
    datasets = config["speakleash-datasets"] + [config["huggingface-dataset-name"]]

    conversations_dir_path = config["dialog-dir-path"]
    preprocessed_conversations_dir_path = config["preprocessed-dialog-path"]
    conversations_datasets = [
        config["conversations-train-dir-name"],
        config["conversations-test-dir-name"],
    ]

    print("Preprocessing written texts ...")
    for dataset in datasets:
        files_names = get_files_names(dir_path=dir_path, dataset_name=dataset)
        preprocess_files(
            dir_path=dir_path,
            preprocessed_dir_path=preprocessed_dir_path,
            dataset_name=dataset,
            files_names=files_names,
        )

    print("Preprocessing conversations transcripts ...")
    for c_dataset in conversations_datasets:
        c_files_names = get_files_names(
            dir_path=conversations_dir_path, dataset_name=c_dataset
        )
        preprocess_files(
            dir_path=conversations_dir_path,
            preprocessed_dir_path=preprocessed_conversations_dir_path,
            dataset_name=c_dataset,
            files_names=c_files_names,
        )


def hyperparameters(config: dict) -> None:
    best_config = run_hyperparameters_optimization(
        config,
        model_config_path=MODEL_CONFIG_PATH,
        save_dir=STUDY_RESULTS_DIR,
        device="cuda" if torch.cuda.is_available() else "cpu",
    )
    print(f"Best hyperparameters saved to {MODEL_CONFIG_PATH}: {best_config}")


def evaluate(config: dict) -> None:
    seq_len = load_config(MODEL_CONFIG_PATH)["seq_len"]
    models = {
        name: LSTMPredictor(
            model_dir=".",
            beam_width=BEAM_WIDTH,
            max_word_length=MAX_WORD_LENGTH,
            device="cpu",
            alpha=ALPHA,
            seq_len=seq_len,
            model_name=model_name,
        )
        for name, model_name in (
            ("lstm", MODEL_PATH),
            ("fine_tuned_lstm", FINE_TUNED_MODEL_PATH),
        )
    }

    test_sets = {
        "written": (
            config["test-indices-file"],
            config["train-paths-file"],
            config["train-offsets-file"],
        ),
        "conversations": (
            config["conversations-test-indices-file"],
            config["conversations-test-paths-file"],
            config["conversations-test-offsets-file"],
        ),
    }

    for test_set, (indices_file, paths_file, offsets_file) in test_sets.items():
        print(f"Evaluating on the {test_set} test set ...")
        output_path = f"evaluation_results_{test_set}.json"
        # ModelEvaluator accumulates results, so each test set needs a new one
        evaluator = ModelEvaluator(models)
        evaluator.evaluate_directory(
            test_indices=np.load(indices_file),
            paths_file=paths_file,
            offsets_file=offsets_file,
            output_path=output_path,
        )
        evaluator.print_comparison()
        evaluator.save_results(output_path)


In [29]:
proj_config = load_config(PROJECT_CONFIG_PATH)

set_seed(proj_config.get("seed", 42))

In [ ]:
%%time

download(proj_config)

In [ ]:
%%time

preprocess(proj_config)

In [ ]:
%%time

create_train_test_split(proj_config)

In [ ]:
%%time

train_tokenizers(proj_config)

In [ ]:
%%time

hyperparameters(proj_config)

In [ ]:
%%time

model_config = load_config(MODEL_CONFIG_PATH)
train_model(proj_config, model_config, save_path=MODEL_CHECKPOINT_PATH)

extract_state_dict(BEST_MODEL_CHECKPOINT_PATH, MODEL_PATH)
shutil.copy(model_config["tokenizer"], TOKENIZER_PATH)

In [ ]:
%%time

fine_tune_model(
    config_dict=proj_config,
    model_path=MODEL_PATH,
    tokenizer_path=TOKENIZER_PATH,
    model_config=load_config(MODEL_CONFIG_PATH),
)

extract_state_dict(BEST_FINE_TUNED_CHECKPOINT_PATH, FINE_TUNED_MODEL_PATH)

In [ ]:
%%time

evaluate(proj_config)